# Distill: fine-tune Qwen3-4B to call a plant copilot's tools

This notebook runs stage 4 end to end on a free Kaggle GPU:

1. **Exam before training**: the base model on the test set, clean and noisy.
2. **QLoRA fine-tuning** on the generated training set.
3. **Exam after training**: the same questions, the same scoring.
4. **Comparison table** for the README, and the LoRA adapter saved.

**Before running:** in the right panel, set *Accelerator* to **GPU T4 x2** (or P100) and turn **Internet** on. To also publish the adapter on the Hugging Face Hub, add a secret named `HF_TOKEN` (*Add-ons → Secrets*).

Then **Run all**. Expect about 2 to 3 hours in total.

In [ ]:
# Install Unsloth and the project. Takes a few minutes.
# Safe to run again: always clones into /kaggle/working/distill, never inside itself.
import os
import sys

REPO = "/kaggle/working/distill"
os.chdir("/kaggle/working")
!pip install -q unsloth
if not os.path.isdir(REPO):
    !git clone -q https://github.com/benothmen12/distill.git {REPO}
os.chdir(REPO)
!pip install -q -e .

# The running kernel does not see a fresh editable install until it restarts, and the
# cloned folder is itself named "distill". Put the package source first on the path.
sys.path.insert(0, os.path.join(REPO, "src"))
for _name in [m for m in sys.modules if m == "distill" or m.startswith("distill.")]:
    del sys.modules[_name]
import distill.data.example  # fails here, early, if the path is still wrong

print("Working folder:", os.getcwd())


In [ ]:
GITHUB_USER = "benothmen12"
DATA_BRANCH = "dataset"  # where the generation workflow saves the data
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"
EVAL_LIMIT = 200  # same stratified subset for every model, clean and noisy
EPOCHS = 2
HF_REPO = "benothmen12/qwen3-4b-distill-lora"  # optional upload

## Data

In [ ]:
import shutil
import urllib.request
from pathlib import Path

data = Path("/kaggle/working/distill/data/processed")
data.mkdir(parents=True, exist_ok=True)
files = ["train.jsonl", "test.jsonl", "test_noisy.jsonl", "stats.json"]

# Option 1: a Kaggle dataset attached to this notebook, with the four files.
attached = [p for p in Path("/kaggle/input").glob("*/train.jsonl")]
if attached:
    for name in files:
        shutil.copy(attached[0].parent / name, data / name)
    print("Data copied from", attached[0].parent)
# Option 2: the dataset branch of the GitHub repo.
else:
    base = f"https://raw.githubusercontent.com/{GITHUB_USER}/distill/{DATA_BRANCH}/processed"
    for name in files:
        urllib.request.urlretrieve(f"{base}/{name}", data / name)
    print("Data downloaded from", base)

print((data / "stats.json").read_text()[:800])

In [ ]:
from distill.data.example import read_examples
from distill.eval.run import evaluate, stratified_sample

train_examples = read_examples("/kaggle/working/distill/data/processed/train.jsonl")
test = stratified_sample(read_examples("/kaggle/working/distill/data/processed/test.jsonl"), EVAL_LIMIT)
test_noisy = stratified_sample(read_examples("/kaggle/working/distill/data/processed/test_noisy.jsonl"), EVAL_LIMIT)
print(f"train: {len(train_examples)}   test: {len(test)}   test_noisy: {len(test_noisy)}")

## 1. Exam before training

The base model, with the same prompt, tools and scoring as every other model.

In [ ]:
import gc
import json

import torch
from unsloth import FastLanguageModel

from distill.eval.local import LocalToolModel
from distill.eval.report import headline_table
from distill.train.format import render
from distill.train.sft import load_model


def exam(model, tokenizer, name):
    tool_model = LocalToolModel(model, tokenizer, name=name, batch_size=8)
    results = {}
    for split, examples in [("test", test), ("test_noisy", test_noisy)]:
        _, metrics, _ = evaluate(examples, tool_model, Path("results") / name / split)
        results[f"{name} ({split})"] = metrics
    print(headline_table(results))
    return results


model, tokenizer = load_model(BASE_MODEL)
FastLanguageModel.for_inference(model)
before = exam(model, tokenizer, "qwen3-4b-base")

### What the model will learn

One training example, rendered with the model's own chat template: the end of the prompt, then the completion. The loss is computed on the completion only.

In [ ]:
row = render(train_examples[0], tokenizer)
print("...", row["prompt"][-500:])
print("======== COMPLETION (what the model learns to write) ========")
print(row["completion"])

del model
gc.collect()
torch.cuda.empty_cache()

## 2. QLoRA fine-tuning

In [ ]:
from distill.train.sft import SFTSettings, make_trainer, run_training, trained_tokens

settings = SFTSettings(base_model=BASE_MODEL, epochs=EPOCHS)
model, tokenizer, trainer, info = make_trainer(train_examples, settings)
print(info["loss_mode"], info["token_lengths"])

# Check the masking: this must be the expected answer only, nothing from the prompt.
print("Trained tokens of example 0:\n", trained_tokens(trainer, tokenizer))

In [ ]:
info = run_training(model, trainer, info)
print(f"train loss {info['train_loss']}, {info['train_runtime_s'] / 60:.0f} min")
Path("results").mkdir(exist_ok=True)
Path("results/train_info.json").write_text(json.dumps(info, indent=2))

## 3. Exam after training

In [ ]:
after = exam(model, tokenizer, "qwen3-4b-sft")

## 4. Comparison

In [ ]:
from distill.eval.compare import collect

table = headline_table(collect([Path("results/qwen3-4b-base"), Path("results/qwen3-4b-sft")]))
Path("results/comparison.md").write_text(table + "\n")
print(table)

sft = after["qwen3-4b-sft (test)"]
print("\nOutcomes after training:", sft["outcomes"])
weakest = sorted(sft["by_field"].items(), key=lambda kv: kv[1]["accuracy"])[:5]
print("Weakest arguments:", weakest)

## 5. Save

The LoRA adapter (about 130 MB) and all results are zipped for download from the *Output* tab. With an `HF_TOKEN` secret, the adapter is also published on the Hugging Face Hub.

In [ ]:
model.save_pretrained("outputs/qwen3-4b-distill-lora")
tokenizer.save_pretrained("outputs/qwen3-4b-distill-lora")
shutil.make_archive("/kaggle/working/distill_results", "zip", ".", "results")
shutil.make_archive("/kaggle/working/distill_lora", "zip", "outputs", "qwen3-4b-distill-lora")

try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(HF_REPO, token=token)
    tokenizer.push_to_hub(HF_REPO, token=token)
    print("Adapter published: https://huggingface.co/" + HF_REPO)
except Exception as exc:  # no secret, or no network: the zip files are still there
    print("Skipping the Hub upload:", exc)